# Local series

Run this notebook from the repository root.

The next cell loads `data/series.npy` when that file exists. The array should be numeric, with one or two dimensions. The figure is saved as `outputs/series.png`. `data/` and `outputs/` are gitignored, and so are `*.npy` files. Large data is not in git. From the repository root, create `data/` and save the array as `data/series.npy`.

That figure is the local file. It is not the Ising trajectory. `evolve` returns `status=scaffold` and no time series.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

series_path = Path("data/series.npy")
figure_path = Path("outputs/series.png")

if not Path("pyproject.toml").is_file():
    print(
        "Run this notebook from the repository root "
        "(the directory that contains pyproject.toml)."
    )
elif not series_path.is_file():
    print(
        "data/series.npy is missing. data/ is gitignored, and so are *.npy files, "
        "so the series is not in git. From the repository root, create data/ and "
        "save a 1D or 2D numeric array as data/series.npy, then run this cell again. "
        "Importing the library does not need that file."
    )
else:
    series = np.load(series_path)
    numeric = isinstance(series, np.ndarray) and np.issubdtype(series.dtype, np.number)
    if not numeric or series.ndim not in (1, 2):
        print(
            f"{series_path.as_posix()} must be a 1D or 2D numeric array. "
            "Replace the file and run this cell again."
        )
    else:
        figure_path.parent.mkdir(parents=True, exist_ok=True)
        fig, ax = plt.subplots()
        ax.plot(series)
        ax.set_xlabel("index")
        ax.set_ylabel("value")
        ax.set_title(series_path.as_posix())
        fig.savefig(figure_path)
        plt.show()
        print(f"Wrote {figure_path.as_posix()}. outputs/ is gitignored.")

## Library scaffold

The next cell composes `conf/` the same way as the tests (`initialize_config_dir` and `compose`) and calls `IsingModel.from_config` and `evolve`. The printed result is a scaffold status. It is not a plot of the dynamics.


In [ ]:
from pathlib import Path

from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra

from QELM.NQS.models import IsingModel

config_dir = Path("conf").resolve()
if not (config_dir / "config.yaml").is_file():
    print("Run this notebook from the repository root so conf/config.yaml resolves.")
else:
    GlobalHydra.instance().clear()
    with initialize_config_dir(version_base="1.3", config_dir=str(config_dir)):
        cfg = compose(config_name="config", overrides=["sim_type=ising"])
    model = IsingModel.from_config(cfg)
    state = model.evolve(dt=float(cfg.test.dt), n_steps=int(cfg.test.n_steps))
    print(
        "scaffold, not a plot of the dynamics: "
        f"model={state['model']} status={state['status']} "
        f"dt={state['dt']} n_steps={state['n_steps']}"
    )